In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score
import yaml
from collections import defaultdict

from data_utils.gecodb_compound_parser import Compound
from data_utils import load_datasets

In [2]:
rng = np.random.default_rng(579372518)

In [ ]:
constr_path = "constraints/constraints_all_revised.yaml"
applind_dir = "resources/semrel_comp_corp_study-private/constr_appl_indices"

# load constraints
with open(constr_path, encoding="utf-8") as f:
    yamls = list(yaml.safe_load_all(f))
    constraints: dict = yamls[-1]["constraints"]
    data_specs: dict = yamls[-2]["datasets"]

# load datasets
datasets = {
    k: getattr(load_datasets, v["load_func"])(v["filepath"])
    for k, v in data_specs.items()
}

# load constraint applicability indices
applinds = {
    k: pd.read_csv(
        os.path.join(applind_dir, f"{k}.tsv"),
        sep="\t",
        header=0,
        index_col=0
    )
    for k in datasets
}

/var/folders/_8/wwb5csy9605g6rl6wf4lcfch0000gr/T/ipykernel_897/2020212830.py:18: DtypeWarning: Columns (0: p2l:decl_cl:pl:#e-0|def-0_item, 1: p2l:decl_cl:pl:#er-0/er|def-0/er_item, 2: p2l:decl_cl:pl:#er-0/er|pl_interpr-er_item, 3: p2l:decl_cl:pl:#e_uml-0|def-0_item, 4: p2l:decl_cl:pl:#0_uml-0|pl_interpr-0_uml_item, 5: p2l:decl_cl:mixed-0/s/en|pl_interpr-en_item, 6: p2l:drv:sfx:e_pl_sfx-0_item, 7: p2l:drv:deverb_schwa$-0|def-0_item, 8: p2l:drv:sfx:sfx-s|def-s_item, 9: p2l:drv:sfx:F_#in$-en_item, 10: p2l:phon_fin:sibilant$-0_item, 11: p2l:phon_fin:F_#stressed_phon$|pl_interpr-en_item, 12: p2l:phon_fin:schwa$-en|def-en_item, 13: p2l:tend:cmpx_morph-s|def-s_item, 14: l2p:en|non_par_n_pl_interpr_item, 15: l2p:e|#stressed_syl_item, 16: l2p:e|smpx_item, 17: l2p:e|!loan_item, 18: l2p:er|#stressed_syl_item, 19: l2p:er|smpx_item, 20: l2p:er|!loan_item, 21: l2p:es|isol_item, 22: l2p:ens_item, 23: l2p:pl_marker|def-pl_item) have mixed types. Specify dtype option on import or set low_memory=False.


In [41]:
# remove unconfirmed and missing constraints
for dname, applind in applinds.items():
    applind.dropna(axis=1, how="all", inplace=True)  # drop columns with NAs
    for c_id, constr in constraints.items():
        if dname not in constr["measures"]:
            continue
        reg_item = constr["measures"][dname]["regularity"]["item"]
        if reg_item < 0.5:
            print(f"Removed {c_id} from {dname} at item reg {reg_item}.")
            applind.drop(
                columns=[
                    f"{c_id}_{sfx}"
                    for sfx in ["item", "is_applicable", "applies"]
                ],
                inplace=True
            )

Removed p2l:decl_cl:pl:#e-0|msyl_anim-e from gecodb_v07_gn at item reg 0.088.
Removed p2l:sem:2const_anim/pers-s from gecodb_v07_gn at item reg 0.482.
Removed p2l:phon_fin:weak_F_cons$-0/s/en|!pl_interpr-0/s from sp18_pl_constr at item reg 0.41.
Removed l2p:pl_marker|def-pl from sp18_pl_constr at item reg 0.146.


In [42]:
applinds["gecodb_v07_gn"].head(3)

,def-0_item,def-0_is_applicable,def-0_applies,p2l:decl_cl:pl:#0-0|def-0_item,p2l:decl_cl:pl:#0-0|def-0_is_applicable,p2l:decl_cl:pl:#0-0|def-0_applies,p2l:decl_cl:pl:#s-0_item,p2l:decl_cl:pl:#s-0_is_applicable,p2l:decl_cl:pl:#s-0_applies,p2l:decl_cl:pl:#e-0|def-0_item,...,l2p:er|smpx_applies,l2p:e_uml_item,l2p:e_uml_is_applicable,l2p:e_uml_applies,l2p:es|mono_syl_item,l2p:es|mono_syl_is_applicable,l2p:es|mono_syl_applies,l2p:0_uml_item,l2p:0_uml_is_applicable,l2p:0_uml_applies
comp_gecodb,,,,,,,,,,,,,,,,,,,,,
land_+es_regierung,land_+es_regierung,True,False,land,False,NaN,land,False,NaN,land,...,NaN,land+es,False,NaN,land+es,True,True,land+es,False,NaN
land_gericht,land_gericht,True,True,land,False,NaN,land,False,NaN,land,...,NaN,land+,False,NaN,land+,False,NaN,land+,False,NaN
land_wirt,land_wirt,True,True,land,False,NaN,land,False,NaN,land,...,NaN,land+,False,NaN,land+,False,NaN,land+,False,NaN


In [ ]:
class CompXConstr:

    def __init__(self, applrow: pd.Series):
        self.comp_gecodb = applrow.name
        self.comp = Compound(applrow.name)
        self.true = self.comp.linkers[0].morph
        self.acceptable_linkers = defaultdict(list)
        # add the default constraint right away
        self.acceptable_linkers["_"].append(constraints["def-0"])
        self.unacceptable_linkers = defaultdict(list)
        self._apply_constraints(applrow)

    def _apply_constraints(self, applrow: pd.Series):
        for c_id, constr in constraints.items():
            if c_id.startswith("p2l:"):
                # acceptable linkers: generated by the P2L constraints
                if applrow.get(f"{c_id}_is_applicable", False):
                    for linker in constr["acceptable_linkers"]:
                        self.acceptable_linkers[linker].append(constr)
            elif c_id.startswith("l2p:"):
                # unacceptable linkers: restricted by the L2P constraints
                pass

In [ ]:
# start with GeCoDB_v07_GN
gecodb = load_gecodb("resources/developed/gecodb_v07_gn.tsv")
N = len(gecodb)

In [21]:
def calc_metrics(golds, preds):
    acc = accuracy_score(golds, preds)
    f1 = f1_score(golds, preds, average="macro")
    return acc, f1

## Baseline: Random Distribution-Aware Sampling

In [10]:
unique_linkers = gecodb["linker"].unique()
linkers = gecodb["linker"].tolist()
linker_distr = pd.Series({
    l: linkers.count(l) / N
    for l in linkers
})
linker_distr.sort_values(ascending=False)

_         0.573485
_+s_      0.268907
_+n_      0.099906
_+en_     0.016232
_+es_     0.015007
_+er_     0.008555
_+e_      0.006452
_+=er_    0.005738
_+ns_     0.003165
_+=e_     0.001940
_+ens_    0.000408
_+=_      0.000204
dtype: float64

In [22]:
rand_linkers = rng.choice(linker_distr.index, size=N, replace=True, p=linker_distr)
calc_metrics(gecodb["linker"], rand_linkers)

(0.4127327670695851, 0.08421396647185787)